# Media-Encrypt Studio v2.0 for Google Colab (USER)
Welcome to the Google Colab runner for Media-Encrypt Studio (v2.0) — **user edition**. This notebook allows you to download your project from GitHub (public repositories) and launch the local Flask server with Colab port forwarding.

### Step 1: Clone the GitHub Repository and Install Dependencies
Enter your public repository link to clone the repository.

In [ ]:
#@title Clone & Setup Repository { display-mode: "form" }
repo_url = "https://github.com/user4534654-gif/media-encrypt/tree/main" #@param {type:"string"}

import os
import shutil

# Parse the branch and base URL if a web interface URL is provided
branch = None
base_url = repo_url

if "/tree/" in repo_url:
    parts = repo_url.split("/tree/")
    base_url = parts[0]
    branch = parts[1]

# Extract the actual repository name (e.g., dev-media-encrypt)
repo_name = base_url.rstrip("/").split("/")[-1].replace(".git", "")

# Clean up previous clone if it exists
if os.path.exists(repo_name):
    print(f"Removing existing directory: {repo_name}")
    shutil.rmtree(repo_name)

# Construct the clone command
if branch:
    print(f"Cloning branch '{branch}' of {base_url}...")
    clone_cmd = f"git clone -b {branch} {base_url}"
else:
    print(f"Cloning default branch of {base_url}...")
    clone_cmd = f"git clone {base_url}"

# Execute the clone command
get_ipython().system(clone_cmd)

# Verify the repository was successfully cloned before changing directories
if os.path.exists(repo_name):
    os.chdir(repo_name)
    print("Installing requirements...")
    !pip install -r requirements.txt
    print("Setup complete!")
else:
    print(f"Error: Directory '{repo_name}' was not created. Please check the repository URL and permissions.")

### Step 2: Run Media-Encrypt Studio
Run this cell to start the server and obtain the interface link. The **tunnel** dropdown offers three options: `colab` (built-in Colab proxy, default), `cloudflare` (Cloudflare Quick Tunnel, no account needed) and `pinggy` (free tunnel, no account needed). Cloudflare and Pinggy URLs also work outside Colab. The cell prints the link before starting the server, so the interface is reachable as soon as the server comes up.

Cloudflared is downloaded as a standalone binary (no pip build), and Pinggy uses its official Python SDK. If a provider is unreachable, the cell falls back to the Colab proxy instead of failing.

A small badge in the bottom-left corner of the interface shows, in the background, whether files picked in the browser actually reached the server's vault. It never blocks the UI.

Use the layout checkboxes in the form below to preview the Desktop (default, unchanged) or Mobile (compact, touch-friendly) scale.

In [ ]:
#@title Run Server { display-mode: "form" }
mode = "tunnel" #@param ["tunnel", "iframe"]
tunnel = "colab" #@param ["colab", "cloudflare", "pinggy"]
use_desktop_layout = True #@param {type:"boolean"}
enable_mobile_layout = False #@param {type:"boolean"}

import os
import re
import shutil
import signal
import subprocess
import sys
import time
import urllib.request
from google.colab import output
from IPython.display import HTML, display

# --- Responsive Web Design layout ---
# Desktop (default) keeps the UI pixel-identical. Mobile activates ONLY when
# its own flag is checked AND the desktop flag is unchecked, so it can never
# turn on by accident. If both flags are checked, desktop wins (safe default).
if enable_mobile_layout and not use_desktop_layout:
    web_layout = "mobile"
else:
    web_layout = "desktop"
    if enable_mobile_layout and use_desktop_layout:
        print("WARNING: both layout flags are checked - desktop wins. "
              "Uncheck 'use_desktop_layout' to preview the mobile scale.")
os.environ["COLAB_DISPLAY_MODE"] = mode
os.environ["WEB_LAYOUT"] = web_layout
print(f"Web layout: {web_layout.upper()}")

PORT = 5050
LOG_PATH = "/tmp/media_encrypt_tunnel.log"
PID_PATH = "/tmp/media_encrypt_tunnel.pid"


def stop_old_tunnel():
    """Stop tunnels left running by an earlier run of this cell."""
    old_sdk_tunnel = globals().get("PINGGY_TUNNEL")
    if old_sdk_tunnel is not None:
        try:
            old_sdk_tunnel.stop()
            print("Stopped previous Pinggy tunnel.")
        except Exception:
            pass
    if os.path.exists(PID_PATH):
        try:
            with open(PID_PATH) as fh:
                old_pid = int(fh.read().strip())
            with open(f"/proc/{old_pid}/cmdline", "rb") as fh:
                cmdline = fh.read().decode("utf-8", "ignore")
            if "cloudflared" in cmdline:
                os.kill(old_pid, signal.SIGTERM)
                print(f"Stopped previous cloudflared tunnel (PID {old_pid}).")
        except (OSError, ValueError, NameError):
            pass
        os.remove(PID_PATH)


def wait_for_url(pattern, label, timeout=90):
    """Watch the tunnel log for its public URL without blocking the notebook."""
    deadline = time.time() + timeout
    last_report = 0.0
    while time.time() < deadline:
        try:
            with open(LOG_PATH, errors="ignore") as fh:
                text = fh.read()
            match = re.search(pattern, text)
            if match:
                return match.group(0).rstrip(".,'\"")
        except OSError:
            pass
        if time.time() - last_report >= 15:
            print(f"Waiting for {label} URL... {int(deadline - time.time())}s left")
            last_report = time.time()
        time.sleep(1)
    print(f"WARNING: no {label} URL appeared within {timeout}s. "
          f"Check the tunnel log: {LOG_PATH}")
    return None


def find_url(value):
    """Extract the first https URL from the Pinggy SDK's urls structure."""
    if isinstance(value, str):
        return value if value.startswith("https://") else None
    if isinstance(value, dict):
        for item in value.values():
            found = find_url(item)
            if found:
                return found
    if isinstance(value, (list, tuple, set)):
        for item in value:
            found = find_url(item)
            if found:
                return found
    return None


def cloudflared_binary():
    """Get a working cloudflared binary without relying on the broken pip wheel."""
    path = shutil.which("cloudflared") or "/tmp/cloudflared"
    if os.path.exists(path):
        try:
            subprocess.check_output([path, "--version"],
                                    stderr=subprocess.STDOUT, timeout=10)
            return path
        except Exception:
            print("Cached cloudflared binary is unusable; downloading again.")
            os.remove(path)
    print("Downloading cloudflared binary (one time)...")
    url = ("https://github.com/cloudflare/cloudflared/releases/latest/download/"
           "cloudflared-linux-amd64")
    try:
        urllib.request.urlretrieve(url, path)
        os.chmod(path, 0o755)
        subprocess.check_output([path, "--version"],
                                stderr=subprocess.STDOUT, timeout=10)
        return path
    except Exception as exc:
        print("Direct download failed:", exc)
        print("Trying pip install cloudflared (errors shown in full)...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "cloudflared"])
        return shutil.which("cloudflared") or path


def colab_proxy():
    return output.eval_js(f"google.colab.kernel.proxyPort({PORT})")


public_url = None
tunnel_name = "Colab proxy"
stop_old_tunnel()

if tunnel == "cloudflare":
    try:
        cf = cloudflared_binary()
        log = open(LOG_PATH, "w")
        proc = subprocess.Popen(
            [cf, "tunnel", "--url", f"http://127.0.0.1:{PORT}"],
            stdout=log, stderr=subprocess.STDOUT)
        with open(PID_PATH, "w") as fh:
            fh.write(str(proc.pid))
        public_url = wait_for_url(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", "Cloudflare")
        tunnel_name = "Cloudflare Quick Tunnel"
    except Exception as exc:
        print(f"Cloudflare tunnel unavailable: {exc}")
elif tunnel == "pinggy":
    try:
        try:
            import pinggy
        except ImportError:
            print("Installing the Pinggy SDK (one time)...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", "pinggy"])
            import pinggy
        # Keep the tunnel object alive for the whole notebook session.
        PINGGY_TUNNEL = pinggy.start_tunnel(
            forwardto=f"localhost:{PORT}", serveraddress="a.pinggy.io:443")
        deadline = time.time() + 45
        while time.time() < deadline and not public_url:
            public_url = find_url(getattr(PINGGY_TUNNEL, "urls", None))
            if not public_url:
                time.sleep(1)
        if not public_url:
            print("Pinggy tunnel started but no URL was reported yet; "
                  "check PINGGY_TUNNEL.urls in this cell.")
        tunnel_name = "Pinggy"
    except Exception as exc:
        print(f"Pinggy tunnel unavailable: {exc}")
else:
    public_url = colab_proxy()

if not public_url:
    public_url = colab_proxy()
    tunnel_name += " (fallback)"

print(f"Tunnel: {tunnel_name}")

if mode == "iframe":
    display(HTML(
        f'<div style="margin-bottom:10px; font-weight:bold; font-family:sans-serif; '
        f'color:#007aff;">Media-Encrypt Studio Interface ({tunnel_name}):</div>'
        f'<iframe src="{public_url}" width="100%" height="800" '
        f'style="border:1px solid #ccc; border-radius:8px; '
        f'box-shadow: 0 4px 12px rgba(0,0,0,0.15);"></iframe>'))
else:
    display(HTML(
        f'<div style="padding:16px; background:#f0f4f8; border-radius:8px; '
        f'border:1px solid #d0d7de; font-family:sans-serif; margin-bottom:12px;">'
        f'<h3 style="margin:0 0 8px 0; color:#0366d6;">'
        f'Media-Encrypt Studio Tunnel Ready</h3>'
        f'<p style="margin:0 0 6px 0; color:#444; font-size:14px;">'
        f'Tunnel: <b>{tunnel_name}</b></p>'
        f'<p style="margin:0 0 12px 0; color:#444; font-size:14px;">'
        f'Click the link below to open the application in a new browser tab:</p>'
        f'<a href="{public_url}" target="_blank" '
        f'style="display:inline-block; padding:10px 20px; background:#007aff; '
        f'color:#fff; text-decoration:none; border-radius:6px; font-weight:bold; '
        f'font-size:14px;">Open Media-Encrypt Studio</a>'
        f'<div style="margin-top:10px; font-size:12px; color:#666; '
        f'font-family:monospace;">{public_url}</div></div>'))

get_ipython().run_line_magic("run", f"main.py --layout {web_layout}")